## Computational Astrobiology – Project Dataset Generation

### Preamble – Importing & Reading

In [1]:
%matplotlib inline
%config InlineBackend.figure_formats = ['retina']

from IPython.display import clear_output

import json
import yaml

import copy
import os
import subprocess
import sys
import time

from pathlib import Path

import tempfile
import h5py

import pandas as pd
import numpy as np
import matplotlib as mpl

import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import h5py

from scipy.stats import truncnorm

from ldtk import LDPSetCreator, BoxcarFilter

plato_filter = BoxcarFilter('plato', 400, 1000)

plt.rcParams['axes.prop_cycle'] = plt.cycler(color=['olivedrab', 'steelblue', 'firebrick', 'goldenrod'])
plt.rcParams['axes.formatter.use_mathtext'] = True
plt.rcParams['axes.formatter.useoffset'] = False
plt.rcParams['axes.formatter.limits'] = (0, 0)
plt.rcParams['figure.figsize'] = [6,4]
plt.rcParams['figure.constrained_layout.use'] = True
plt.rcParams['legend.frameon'] = False
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['ytick.minor.visible'] = True

template_file = 'PSLS/examples/psls.yaml'

with open(template_file, 'r') as file:
    config_dict = yaml.safe_load(file)
    print(json.dumps(config_dict, indent=4, sort_keys=False))

{
    "Observation": {
        "QuarterDuration": [
            90.0,
            90.0,
            90.0
        ],
        "MasterSeed": 1704040900,
        "Gaps": {
            "Enable": 1,
            "Seed": -1,
            "InterQuarterGapDuration": 3.0,
            "RandomGapDuration": 0.0,
            "RandomGapTimeFraction": 0.5,
            "RandomGapStep": 0.0,
            "PeriodicGapCadence": 5.0,
            "PeriodicGapDuration": 20.0,
            "PeriodicGapJitter": 2.0,
            "PeriodicGapStep": 0.0
        }
    },
    "Instrument": {
        "Sampling": 25.0,
        "IntegrationTime": 21.0,
        "GroupID": [
            1,
            2,
            3,
            4
        ],
        "NCamera": 6,
        "TimeShift": 6.25,
        "RandomNoise": {
            "Enable": 1,
            "Type": "PLATO_SIMU",
            "NSR": 73.0
        },
        "Systematics": {
            "Enable": 1,
            "Table": "systematics/PLATO_systematics_BOL_V2.npy",
  

### Template Dictionary

In [2]:
psls_template = {
    'Observation': {
        'QuarterDuration': [90.0] * 16,     # days
        'MasterSeed': 0,
        'Gaps': {
            'Enable': 1,
            'Seed': -1,
            'InterQuarterGapDuration': 3.0, # days
            'RandomGapDuration': 20.0,      # minutes
            'RandomGapTimeFraction': 0.05,
            'RandomGapStep': 0.0,
            'PeriodicGapCadence': 5.0,      # days
            'PeriodicGapDuration': 20.0,    # minutes
            'PeriodicGapJitter': 2.0,
            'PeriodicGapStep': 0.0
        }
    },
    'Instrument': {
        'Sampling': 300.0,                 # seconds (default: 25.0)
        'IntegrationTime': 252.0,          # seconds (default: 21.0)
        'GroupID': [1, 2, 3, 4],
        'NCamera': 6,
        'TimeShift': 6.25,                 # seconds
        'RandomNoise': {
            'Enable': 1,
            'Type': 'PLATO_SIMU',
            'NSR': 73.0
        },
        'Systematics': {
            'Enable': 1,
            'Table': 'systematics/PLATO_systematics_BOL_V2.npy',
            'Version': 2,
            'DriftLevel': 'any',
            'Seed': -1
        }
    },
    'Star': {
        'Mag': 10.0,
        'ID': 0,
        'ModelType': 'grid',
        'ModelDir': 'models/',
        'ModelName': 'grid.hdf5',
        'ES': 'ms',
        'Teff': 5778.0,                   # not UP
        'Logg': 4.438,                    # not UP
        'SurfaceRotationPeriod': 0.0,     # not UP
        'CoreRotationFreq': 0.0,          # only UP
        'Inclination': 0.0
    },
    'Oscillations': {
        'Enable': 1,
        'numax': 180.0,                    # only UP
        'delta_nu': 14.0,                  # only UP
        'DPI': 81.0,                       # only UP
        'q': 0.15,                         # only UP
        'SurfaceEffects': 1,               # not UP
        'Seed': -1
    },
    'Activity': {
        'Enable': 1,
        'Sigma': 40.0,
        'Tau': 0.2,
        'Seed': -1,
        'Spot': {
            'Enable': 0,
            'dOmega': 0.0,
            'MuStar': 0.6,
            'MuSpot': 0.8,
            'Radius': [2.5, 2.5, 2.5],
            'Latitude': [0.0, 20.0, 40.0],
            'Longitude': [0.0, 0.0, 0.0],
            'Lifetime': [10, 30, 50],
            'TimeMax': [-1, -1, -1],
            'Contrast': [0.7, 0.8, 0.6],
            'Modulation': 0.0,
            'Seed': -1
        },
        'Flare': {
            'Enable': 0,
            'MeanPeriod': 2,
            'Amplitude': 2500.0,
            'UpDown': 0.1,
            'MeanDuration': -1,
            'DurationDispersion': -1,
            'Seed': -1
        }
    },
    'Granulation': {
        'Enable': 1,
        'Type': 1,
        'Seed': -1
    },
    'Transit': {
        'Enable': 1,
        'PlanetRadius': 0.5,
        'OrbitalPeriod': 10.0,
        'PlanetSemiMajorAxis': 1.0,
        'OrbitalAngle': 0.0,
        'LimbDarkeningCoefficients': [0.25, 0.75]
    },
    'External': {
        'Enable': 0,
        'FilePath': 'examples/external_example.txt'
    }
}

### Auxiliary Functions

In [3]:
def initialize(file_name, total_sims, date_string):
    with h5py.File(file_name, 'x') as f:
        f.attrs['description'] = f'Batch Simulation Results ({total_sims} Runs)'
        f.attrs['date_created'] = date_string
        f.attrs['user'] = 'Fritz Ali Agildere'

def run(temp_file):
    tmp_file = Path(temp_file.name).resolve()
    work_dir = Path('PSLS')
    curr_cmd = [sys.executable, 'psls.py', '-V', '-o', 'data', str(tmp_file)]

    subprocess.run(
        curr_cmd,
        cwd=work_dir,
        input='\n',
        text=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False
    )

def save(i, file_name, group_name, temp_file):
    with h5py.File(file_name, 'a') as f:
        run = f.create_group(group_name)
        
        with open(temp_file.name, 'r') as cfg:
            run.attrs['config_used'] = cfg.read()

        with open(f'PSLS/data/{i:010d}.txt', 'r') as txt:
            content = txt.read()
            dt = h5py.string_dtype(encoding='utf-8')
            run.create_dataset('txt', data=content, dtype=dt)
            
        dat = f[group_name].create_group('dat')
        time_s, flux_var_ppm, flag = np.genfromtxt(f'PSLS/data/{i:010d}.dat', unpack=True, skip_header=5)
        
        dat.create_dataset('time_s', data=time_s, compression='gzip', shuffle=True)
        dat.create_dataset('flux_var_ppm', data=flux_var_ppm, compression='gzip', shuffle=True)
        dat.create_dataset('flag', data=flag, compression='gzip', shuffle=True)
        
        modes = f[group_name].create_group('modes')
        n, l, m, nu, gamma, h, I_Imax, dnu, split = np.genfromtxt(f'PSLS/data/{i:010d}.modes', unpack=True, skip_header=1)
        
        modes.create_dataset('n', data=n, compression='gzip', shuffle=True)
        modes.create_dataset('l', data=l, compression='gzip', shuffle=True)
        modes.create_dataset('m', data=m, compression='gzip', shuffle=True)
        modes.create_dataset('nu', data=nu, compression='gzip', shuffle=True)
        modes.create_dataset('gamma', data=gamma, compression='gzip', shuffle=True)
        modes.create_dataset('h', data=h, compression='gzip', shuffle=True)
        modes.create_dataset('I_Imax', data=I_Imax, compression='gzip', shuffle=True)
        modes.create_dataset('dnu', data=dnu, compression='gzip', shuffle=True)
        modes.create_dataset('split', data=split, compression='gzip', shuffle=True)

def clean(i):
    txt = f'PSLS/data/{i:010d}.txt'
    dat = f'PSLS/data/{i:010d}.dat'
    modes = f'PSLS/data/{i:010d}.modes'
    
    %rm {txt}
    %rm {dat}
    %rm {modes}

### Parameter Sampling

In [4]:
# --- defining constants --- #

teff_sun = 5.778e3
m_sun = 1.98919e33
r_sun = 6.9599e10
lum_sun = 3.828e33
logg_sun = 4.438e0
rot_sun = 2.6e1
sig_sun = 3e1

r_earth = 9.17e-2
p_earth = 3.65256e2 

def sample(i, config_dict, planet_setting, use_case, rng):
    
    # --- simulation setup --- #

    config_dict['Observation']['MasterSeed'] = i
    config_dict['Star']['ID'] = i
    config_dict['Star']['Mag'] = float(rng.uniform(9.0, 13.0))

    # --- stellar inclination truncated normal distribution --- #
    
    mean = 90.0
    std = 15.0
    lower = 0.0
    upper = 90.0
    
    a = (lower - mean) / std
    b = (upper - mean) / std
    
    i = float(truncnorm.rvs(a, b, loc=mean, scale=std, size=1, random_state=rng))
    
    config_dict['Star']['Inclination'] = i

    # --- metallicity random parameter --- #

    metal = float(rng.normal(-0.1, 0.2))

    # --- stellar parameter setup --- #
    
    if use_case == 'training':
        mass = float(rng.uniform(0.91, 1.19))
    elif use_case == 'science':
        mass = float(rng.uniform(1.2**(-1.35), 0.9**(-1.35))**(-1/1.35))

    # --- calculating physical values  --- #
    
    radius = float(mass**0.8 * (1 + 0.12 * metal))
    teff = float(teff_sun * mass**0.5 * (1 - 0.06 * metal))
    logg = float(logg_sun + np.log10(mass) - 2 * np.log10(radius))
    lum = float(lum_sun * radius**2 * (teff / teff_sun)**4)
    
    config_dict['Star']['Teff'] = teff
    config_dict['Star']['Logg'] = logg

    # --- decide rotation type --- #

    if use_case == 'training':
        rotType = rng.choice(['fast', 'slow'], p=[0.5, 0.5])
    elif use_case == 'science':
        rotType = rng.choice(['fast', 'slow'], p=[0.05, 0.95])

    if rotType == 'slow':
        rot_period = float(rng.lognormal(np.log(25), 0.2))
    elif rotType == 'fast':
        rot_period = float(rng.lognormal(np.log(4), 0.4))

    config_dict['Star']['SurfaceRotationPeriod'] = rot_period

    # --- choose activity parameters based on rotation --- #

    config_dict['Activity']['Sigma'] = float(rng.normal(sig_sun, sig_sun / 6)) * (rot_sun / rot_period)**2
    config_dict['Activity']['Tau'] = rot_period / 2 + float(rng.normal(0, rot_period / 20))

    tau_convect = 14 * mass**(-2.5)
    ratio = rot_period / tau_convect
    sigmoid = float(1 / (1 + np.exp(4 * (ratio - 0.8))))

    config_dict['Activity']['Spot']['Enable'] = int(rng.uniform(0, 1) < sigmoid)
    config_dict['Activity']['Flare']['Enable'] = int(rng.uniform(0, 1) < sigmoid)

    num = int(rng.integers(12, 48) / rot_period)

    config_dict['Activity']['Spot']['Radius'] = [2.5] * num
    config_dict['Activity']['Spot']['Latitude'] = [float(rng.normal(20, 5)) * float(rng.choice([-1, 1])) for _ in range(num)]
    config_dict['Activity']['Spot']['Longitude'] = [float(rng.uniform(0, 100)) if i < num//2 else float(rng.uniform(180, 280)) for i in range(num)]
    config_dict['Activity']['Spot']['Lifetime'] = [rot_period * float(rng.uniform(1, 3)) for _ in range(num)]
    config_dict['Activity']['Spot']['Contrast'] = [0.75] * num
    config_dict['Activity']['Spot']['TimeMax'] = [-1] * num

    config_dict['Activity']['Flare']['MeanPeriod'] = 2.0 * (rot_period / 10)**2
    config_dict['Activity']['Flare']['Amplitude'] = 2500.0 * (10 / rot_period)

    config_dict['Activity']['Flare']['MeanDuration'] = float(rng.uniform(0.02, 0.12))
    config_dict['Activity']['Flare']['DurationDispersion'] = 0.5 * (rot_period / 10)**2

    # --- planet settings adjustment --- #

    if planet_setting == 'on':
        config_dict['Transit']['Enable'] = 1

        if use_case == 'training':
            r_planet = r_earth * 10.0**float(rng.uniform(np.log10(0.4), np.log10(3.6)))
            seff = 10.0**float(rng.uniform(np.log10(0.16), np.log10(16.0)))
            normlum = (radius**2) * ((teff / teff_sun)**4)
            a_planet = float(np.sqrt(normlum / seff))
        elif use_case == 'science':
            choice = rng.choice(['rock', 'gas'])
            if choice == 'rock':
                r_planet = r_earth * float(rng.normal(1.3, 0.3))
            elif choice == 'gas':
                r_planet = r_earth * float(rng.normal(2.4, 0.4))
            a_planet = float(rng.normal(1.1, 0.3))
        
        p_planet = p_earth * (a_planet**3 / mass)**(1/2)

        config_dict['Transit']['PlanetRadius'] = r_planet
        config_dict['Transit']['PlanetSemiMajorAxis'] = a_planet
        config_dict['Transit']['OrbitalPeriod'] = p_planet
        config_dict['Transit']['OrbitalAngle'] = float(rng.uniform(-0.5, 0.5))

        orig_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')
        try:
            sc = LDPSetCreator(teff=(teff, 50), logg=(logg, 0.05), z=(metal, 0.1), filters=[plato_filter])
            ps = sc.create_profiles()
            u1, u2 = ps.coeffs_qd()
        finally:
            sys.stdout.close()
            sys.stdout = orig_stdout

        config_dict['Transit']['LimbDarkeningCoefficients'] = [float(u1[0,0]), float(u2[0,0])]
        
    elif planet_setting == 'off':
        config_dict['Transit']['Enable'] = 0

### Simulation Loop

In [5]:
def simulate(total_sims = 100, initial_state = 0, seed = 123, title = 'none', usage = 'science', planet = 'on'):
    
    start_batch = time.time()

    id_str = time.strftime('%Y%m%d%H%M%S', time.gmtime(start_batch))
    attr_str = time.strftime('%d.%m.%Y, %H:%M:%S', time.gmtime(start_batch))

    if title != 'none':
        file_name = f'PSLS/data/{title}.hdf5'
    else:
        file_name = f'PSLS/data/{id_str}.hdf5'

    # --- initializing dataframe file --- #

    initialize(file_name, total_sims, attr_str)

    initial_status = f'│ progress |{'░' * 50}| 0% [running 1/{total_sims}] (00:00:00/--:--:--)          '
    
    print(f'{initial_status}')
    
    i = initial_state
    j = 0
    k = h5py.File(file_name, 'r')
    while len(k) < total_sims:
        k.close()

        if i != initial_state:
            perc = (i - initial_state) / (total_sims + j) * 100
            bar = '█' * int(perc // 2) + '░' * (50 - int(perc // 2))
            status = f'│ progress |{bar}| {perc:.0f}% [running {i - initial_state + 1}/{total_sims + j}] ({elapsed_str}/{estimated_str})          '
    
            clear_output(wait=True)
            print(f'{status}')

        # --- baseline template --- #

        config = copy.deepcopy(psls_template)

        sys.stdout.write(f'└── initializing simulation {i + 1}          ')
        sys.stdout.flush()
        sys.stdout.write(f'\r└─┬ initializing simulation {i + 1}          \n')
        sys.stdout.write(f'  └── sampling parameters [active]          ')
        sys.stdout.flush()

        # --- sampling distributions based on use case --- #

        sample(i, config, planet, usage, np.random.default_rng(i + j + seed))
        
        with tempfile.NamedTemporaryFile(suffix='.yaml', dir='PSLS', mode='w', delete=True) as tf:
            yaml.dump(config, tf, default_flow_style=False, sort_keys=False)
            tf.flush()
            
            sys.stdout.write(f'\r  ├── sampling parameters [done]          \n')
            sys.stdout.write(f'  └── generating lightcurve [active]          ')
            sys.stdout.flush()

            # --- running the simulator --- #

            run(tf)

            if os.path.isfile(f'PSLS/data/{i:010d}.txt'):
                sys.stdout.write(f'\r  ├── generating lightcurve [done]          \n')
                sys.stdout.write(f'  └── saving dataframe [active]          ')
                sys.stdout.flush()

                group_name = f'run_{i + 1:05d}'

                # --- storing the file --- #

                save(i, file_name, group_name, tf)
                clean(i)
            else:
                j += 1

        current_run = time.time()
        
        elapsed_batch = current_run - start_batch
        
        average_time = elapsed_batch / (i - initial_state + 1)
        
        estimated_batch = elapsed_batch + (total_sims + j - (i - initial_state + 1)) * average_time
    
        elapsed_str = time.strftime('%H:%M:%S', time.gmtime(elapsed_batch))
        estimated_str = time.strftime('%H:%M:%S', time.gmtime(estimated_batch))

        i += 1
        k = h5py.File(file_name, 'r')

    final_status = f'│ progress |{'█' * 50}| 100% [finished {i - initial_state}/{total_sims + j}] ({elapsed_str}/{estimated_str})          '

    clear_output(wait=True)
    print(f'{final_status}')

    return initial_state + total_sims + j

In [6]:
Path('PSLS/data/example.hdf5').unlink(missing_ok=True)
exampleState = simulate(title='example', total_sims=10, seed=321)

│ progress |██████████████████████████████████████████████████| 100% [finished 17/17] (00:00:49/00:00:49)          


In [7]:
# trainPlanetState = simulate(title='trainPlanetFix', total_sims=500, initial_state=0, usage='training', planet='on')

│ progress |██████████████████████████████████████████████████| 100% [finished 667/667] (04:31:03/04:31:03)          


In [8]:
# trainStarState = simulate(title='trainStarFix', total_sims=500, initial_state=trainPlanetState, usage='training', planet='off')

│ progress |██████████████████████████████████████████████████| 100% [finished 679/679] (04:00:05/04:00:05)          


In [9]:
# realPlanetState = simulate(title='realPlanetFix', total_sims=150, initial_state=trainStarState, usage='science', planet='on')

│ progress |██████████████████████████████████████████████████| 100% [finished 220/220] (00:20:02/00:20:02)          


In [10]:
# realStarState = simulate(title='realStarFix', total_sims=350, initial_state=realPlanetState, usage='science', planet='off')

│ progress |██████████████████████████████████████████████████| 100% [finished 475/475] (00:43:04/00:43:04)          
